# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*
Research question: Among a client's existing content pages, which ones are
at elevated risk of search-traffic decline in the near term, and what should
a content team do about each one?

Decision it supports: weekly content-refresh triage. A content team with
limited review time each week needs a ranked list of pages to look at first,
with a stated reason for each -- not a fully-automated republish pipeline.
This is decision-support for humans, not an autonomous action system (see
Limitations and the no-go list carried over from w07).

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*
Release: the 30,000-row starter CSV (data/raw/content_refresh_anonymized.csv),
32 pseudonymized clients, one row per content item, all metrics trailing-90-day
as of export. Not the full ~79M-row warehouse release -- this capstone stays
on the starter slice for a reproducible, single-file analysis.

Population filter applied: impressions_90d > 0 AND content_age_days >= 90 --
excludes brand-new and zero-traffic pages. This is a design choice about
which pages are in scope for refresh triage (a page with zero impressions
isn't a refresh candidate, it's an indexing/visibility problem), disclosed
here per the leakage skill's rule that filters need naming, not hiding.

Excluded columns and why: trend_direction/trend_pct (label source),
impressions_last_30d/prev_30d and clicks/sessions equivalents (label's own
comparison windows), content_id/client_id (grouping only), provider_used/
model_used (data dictionary marks these "not a model feature"). Full list
and reasoning in w03_feature_leakage_check.ipynb Section 4.

Public-safe: no client names, no URLs, no raw query text anywhere in this
notebook or its outputs -- IDs are pseudonyms throughout.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*
Label: is_declining_label = 1 when trend_direction == "down" (last-30d
impressions down >20% vs. the prior 30d), else 0. Base rate: 54.2%.

Features: MODEL_NUMERIC_FEATURES (18 cols: keyword metadata, content
properties, log-transformed trailing-90d traffic totals, derived rates)
+ MODEL_CATEGORICAL_FEATURES (8 cols, one-hot encoded) from
scripts/ml_utils.py -- the single canonical feature list reused across
every notebook in this repo. Full feature-by-feature notes (meaning,
missing handling, available-when) in w03_feature_leakage_check.ipynb
Section 2.

Baseline: a hand-written rule from w04_baseline_score.ipynb --
freshness_tier == "91-180" AND impressions_90d >= 300 -> flag for refresh.
Measured decline rate: 61.7% flagged vs. 51.8% not flagged vs. 54.2%
base rate (w04_signal_audit.ipynb Section 3) -- a real, floor-clearing
lift (n=7,212), but modest.

Model: Random Forest (class_weight="balanced_subsample", max_depth=10,
min_samples_leaf=25, 200 trees), chosen over Logistic Regression and a
Decision Tree per training-honest-models/SKILL.md's question-shape table
("yes/no with an observed label" -> Logistic Regression then Random
Forest) and because several features showed threshold-like, non-linear
relationships with decline in the signal audit (position_tier and
freshness_tier both broke strict monotonicity -- w04_signal_audit.ipynb
Section 2) that a linear model can't capture.

Validation design: grouped client-holdout split (20% of 32 clients, 6
held out entirely, seeded random_state=42) -- client_id used for
grouping only, never as a feature, per flyrank-data/SKILL.md.

Leakage checks: full attack checklist run in w03_feature_leakage_check.ipynb
and repeated in w06_validation_audit.ipynb -- injecting the label-derived
trend_pct as a feature collapses ROC AUC from 0.750 (honest) to a perfect
1.000 with the injected column claiming 82.7% of feature importance,
confirming both that the real feature set is clean and that the test
harness actually catches leakage when present.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*
Same held-out client split, same feature set, compared using precision@K
(the repo's own established ranking metric) rather than a raw binary
threshold -- because the baseline rule's own threshold flagged only 11
of 2,325 test rows on this particular 6-client holdout (recall 0.007),
too few to compare reliably at a single cutoff (w06_validation_audit.ipynb
Section 2).

| K   | Rule (baseline) | Random Forest |
|-----|------------------|----------------|
| 20  | 0.55             | **0.65**       |
| 50  | 0.50             | **0.74**       |
| 100 | 0.42             | **0.72**       |

ROC AUC: 0.750 (Random Forest). Average precision: 0.618.

Also measured: a random (non-grouped) split on the same model gives ROC
AUC 0.758 and avg_precision 0.768 -- both HIGHER than the honest grouped
numbers above, because 31 of 32 clients overlap between train and test
under random splitting. Reporting both, per hunting-leakage-and-validating/
SKILL.md's instruction to report the gap as a finding in itself
(w06_validation_audit.ipynb Section 2).

Of the 903 test pages that actually declined and that the rule's
threshold missed, Random Forest recovered 670 (74.2%) of them
(w05_model.ipynb Section 4).

## 5. Limitations

*What this work cannot claim.* - Cross-sectional data, one snapshot: this cannot support "refreshing a
  page WILL improve its ranking" -- only "these pages look worth
  reviewing first" (decision-support, per writing-honest-claims/SKILL.md's
  claim ladder)
- Client-holdout variance: with only 32 clients, which 6 land in the test
  set matters -- the baseline rule's own flag count on this split (11 rows)
  shows how thin a single holdout can be; a different holdout could shift
  precision@K meaningfully
- The model cannot separate content-quality decline from external causes
  (competitor moves, SERP feature changes, seasonality) -- flagged directly
  in w07_action_playbook.ipynb's human-review section
- Population filter excludes brand-new and zero-traffic pages by design --
  this analysis says nothing about those pages
- Starter 30K-row slice only, not the full warehouse -- generalization to
  the full client portfolio is untested here
- decline_probability is a risk ranking for prioritization, not a
  prediction of what any search algorithm will do

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.* Output of w07_action_playbook.ipynb, regenerated here for the paper
(queue CSV stays out of git by design -- see that notebook's Section 5
note on the CI leak-guard; the committed metrics JSON and figure are the
receipts).

Action mix across 30,000 scored pages: monitor 56.7% (17,009), refresh
31.9% (9,569), refresh_and_review_ctr 11.1% (3,340), expand_and_refresh
0.3% (82).

Highest-risk archetype (n>=50 floor): freshness_tier 91-180 x position_tier
top_3 -- 67.9% observed decline rate (n=302), the single riskiest
combination measured. Notably, staleness alone is a much weaker signal:
the same freshness_tier crossed with position_tier "deep" shows only
35.9% decline (n=312) -- position interacts with freshness rather than
freshness acting alone, which is why the ranked queue uses the model's
joint probability rather than either dimension in isolation.

Cost/value framing: review cost is roughly constant per page regardless
of traffic, so the queue reports decline_probability alongside
impressions_90d (not multiplied into one score, to keep it auditable) --
a time-limited reviewer should scan high-impression, high-probability
rows first.

In [1]:
# Regenerate the ranked queue for the paper (full pipeline, matches w07 exactly)
import pandas as pd, numpy as np, json
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
df = pd.read_csv('https://raw.githubusercontent.com/Iqra411/lyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv')
numeric_fill_zero = ["search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","ai_sessions_90d","scroll_events_90d","days_with_impressions",
    "days_with_sessions","impressions_last_30d","clicks_last_30d","sessions_last_30d",
    "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","content_age_days",
    "age_tier_order","days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct","trend_pct"]
for c in numeric_fill_zero:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
cat_cols = ["competition_level","content_type","main_intent","provider_used","model_used",
    "age_tier","freshness_tier","word_count_tier","char_count_tier","impression_tier",
    "position_tier","trend_direction"]
for c in cat_cols:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
for col, src in [("log_impressions_90d","impressions_90d"),("log_clicks_90d","clicks_90d"),
                  ("log_sessions_90d","sessions_90d"),("log_ai_sessions_90d","ai_sessions_90d")]:
    df[col] = np.log1p(df[src])

MODEL_NUMERIC_FEATURES = ["search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
MODEL_CATEGORICAL_FEATURES = ["competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier"]
num_frame = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
cat_frame = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES].astype(str), prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
feat = pd.concat([num_frame.reset_index(drop=True), cat_frame.reset_index(drop=True)], axis=1)
target = df["is_declining_label"].astype(int)

rf_full = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf_full.fit(feat, target)
df["decline_probability"] = rf_full.predict_proba(feat)[:, 1]

def reason_codes(row):
    r = []
    if row["decline_probability"] >= 0.65: r.append("model_high_decline_risk")
    if row["freshness_tier"] == "91-180" and row["impressions_90d"] >= 300: r.append("stale_but_visible")
    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250: r.append("thin_visible_page")
    if row["position_tier"] in ("page_1","top_3") and row["decline_probability"] >= 0.5: r.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5: r.append("low_ctr_visible_page")
    return "|".join(r) if r else "general_review"
df["reason_codes"] = df.apply(reason_codes, axis=1)

def action(row):
    r = set(row["reason_codes"].split("|"))
    if "thin_visible_page" in r: return "expand_and_refresh"
    if "low_ctr_visible_page" in r and "model_high_decline_risk" in r: return "refresh_and_review_ctr"
    if "model_high_decline_risk" in r or "stale_but_visible" in r: return "refresh"
    return "monitor"
df["suggested_action"] = df.apply(action, axis=1)
print(df["suggested_action"].value_counts())

arch = df.groupby(["freshness_tier","position_tier"], observed=True).agg(
    n=("content_id","size"), decline_rate=("is_declining_label","mean")).reset_index()
print(arch[arch["n"]>=50].sort_values("decline_rate", ascending=False).head(5))

suggested_action
monitor                   17009
refresh                    9569
refresh_and_review_ctr     3340
expand_and_refresh           82
Name: count, dtype: int64
   freshness_tier position_tier     n  decline_rate
18         91-180         top_3   302      0.678808
11          31-90      page_3_5    53      0.660377
17         91-180      striking  2338      0.636869
15         91-180        page_1  3335      0.623088
3            0-30      striking  4862      0.597491


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [2]:
import os
os.makedirs("work/figures", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Figure 1: precision@K comparison
fig, ax = plt.subplots(figsize=(6,4))
ks = [20,50,100]
rule_vals = [0.55,0.50,0.42]
rf_vals = [0.65,0.74,0.72]
ax.plot(ks, rule_vals, marker='o', label='Baseline rule')
ax.plot(ks, rf_vals, marker='o', label='Random Forest')
ax.set_xlabel("K"); ax.set_ylabel("Precision@K"); ax.legend(); ax.set_title("Model vs. baseline, honest split")
plt.tight_layout(); plt.savefig("work/figures/precision_at_k.png", dpi=150); plt.close()

# Figure 2: action mix
plt.figure(figsize=(7,4))
df["suggested_action"].value_counts().plot(kind="barh")
plt.xlabel("Pages"); plt.title("Suggested action mix")
plt.tight_layout(); plt.savefig("work/figures/action_mix.png", dpi=150); plt.close()

# Metrics JSON -- the paper's numeric receipts
paper_metrics = {
    "base_rate": 0.542,
    "baseline_rule": {"flagged_n": 7212, "flagged_decline_rate": 0.617, "unflagged_decline_rate": 0.518},
    "model_grouped_split": {"roc_auc": 0.750, "avg_precision": 0.618,
        "precision_at_20": 0.65, "precision_at_50": 0.74, "precision_at_100": 0.72},
    "model_random_split_for_comparison": {"roc_auc": 0.758, "avg_precision": 0.768,
        "precision_at_20": 0.95, "precision_at_50": 0.90, "precision_at_100": 0.90},
    "rule_missed_model_caught": {"rule_missed": 903, "model_recovered": 670, "pct": 0.742},
    "action_mix": df["suggested_action"].value_counts().to_dict(),
    "note": "All figures are decision-support, not causal claims."
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(paper_metrics, f, indent=2, default=str)
print("Wrote work/figures/precision_at_k.png, work/figures/action_mix.png, work/outputs/capstone_metrics.json")

Wrote work/figures/precision_at_k.png, work/figures/action_mix.png, work/outputs/capstone_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
